In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv("LandlordData_9_1_26.csv")
df.drop(columns=["Username", "Unnamed: 10"]).head()

,Landlord,Address (if mentioned),Source,URL,Date Posted,Stars (if applicable),Review Text,Pos or Neg Review?,Date Recorded
0,Wolfe & Associates,NaN,Google,https://www.google.com/search?q=wolfe+and+asso...,7/2/26,5.0,"Easy to understand, streamlined application pr...",Positive,7/16/2026
1,Wolfe & Associates,NaN,Google,https://www.google.com/search?q=wolfe+and+asso...,6/28/26,5.0,"The team was set up and ready for check in, wi...",Positive,7/16/2026
2,Wolfe & Associates,NaN,Google,https://www.google.com/search?q=wolfe+and+asso...,6/26/26,5.0,So impressed. Had such a horrible experience w...,Positive,7/16/2026
3,Wolfe & Associates,NaN,Google,https://www.google.com/search?q=wolfe+and+asso...,6/26/26,4.0,The moving in process was easy. The apartment ...,Positive,7/16/2026
4,Wolfe & Associates,NaN,Google,https://www.google.com/search?q=wolfe+and+asso...,6/19/26,5.0,I’m so thankful that my daughter was able to f...,Positive,7/16/2026


In [3]:
#Quick check of how data looks

print(f"Total rows: {len(df)}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nMissing values per column:\n{df.isna().sum()}")

Total rows: 1004

Columns: ['Landlord', 'Address (if mentioned)', 'Source', 'URL', 'Date Posted', 'Stars (if applicable)', 'Username', 'Review Text', 'Pos or Neg Review?', 'Date Recorded', 'Unnamed: 10']

Missing values per column:
Landlord                    0
Address (if mentioned)    999
Source                      0
URL                         0
Date Posted                 0
Stars (if applicable)      34
Username                    0
Review Text                 0
Pos or Neg Review?          0
Date Recorded               0
Unnamed: 10               996
dtype: int64


In [4]:
#Unique values for each relevant column
for col in ["Landlord", "Source", "Stars (if applicable)", "Pos or Neg Review?", "Date Recorded"]:
    print(f"--- {col} ---")
    print(df[col].unique())
    print()

df["Pos or Neg Review?"].unique()

--- Landlord ---
<StringArray>
[                   'Wolfe & Associates',
            'Sierra Property Management',
 'Meridian Group Real Estate Management',
                            'Koto Group',
                  'Silverwood Townhomes',
                  'Breakpointe Coronado',
                    'Bartlein & Company',
                              'Capri IV',
                   'State Santa Barbara']
Length: 9, dtype: str

--- Source ---
<StringArray>
[              'Google',                 'Yelp', 'Yelp (hidden review)',
     'Rate My Landlord',          'BBB Reviews',       'BBB Complaints',
             'Facebook',             'RentCafe',     'ApartmentRatings']
Length: 9, dtype: str

--- Stars (if applicable) ---
[5.  4.  1.  2.  3.  nan 4.5 3.5 2.5 1.2 4.8 1.3]

--- Pos or Neg Review? ---
<StringArray>
['Positive', 'Negative', 'Mixed']
Length: 3, dtype: str

--- Date Recorded ---
<StringArray>
['7/16/2026', '7/17/2026', '7/18/2026', '7/19/2026', '7/20/2026', '7/22/2026',
 '7

<StringArray>
['Positive', 'Negative', 'Mixed']
Length: 3, dtype: str

In [5]:
#Any duplicate review texts (although this is fine, this is just exploratory)

dupes = df[df.duplicated(subset=["Review Text"], keep=False)].sort_values("Review Text")
dupes[["Landlord", "Username", "Source", "Review Text"]]

,Landlord,Username,Source,Review Text
566,Koto Group,Maritza M.,Yelp,Does anyone who is the person behind this mana...
498,Koto Group,mari o,Google,Does anyone who is the person behind this mana...
574,Koto Group,Dana C.,Yelp,I moved into Koto Group's property in San Luis...
544,Koto Group,Dana Casabella,Google,I moved into Koto Group's property in San Luis...
785,Breakpointe Coronado,Alex P.,Yelp,I started living at Breakpointe last summer an...
715,Breakpointe Coronado,alex,Google,I started living at Breakpointe last summer an...
48,Wolfe & Associates,Dylan Apsey,Google,"Negative\r\nResponsiveness, Quality, Professio..."
51,Wolfe & Associates,Daniel ahmadian,Google,"Negative\r\nResponsiveness, Quality, Professio..."
539,Koto Group,Gerald Boyle,Google,"Positive\nResponsiveness, Professionalism"
217,Sierra Property Management,Melissa Camphous,Google,"Positive\nResponsiveness, Professionalism"


In [6]:
#Make sure reviews are from 2020 on
df["Date Posted"] = pd.to_datetime(df["Date Posted"], errors="coerce")
print(df["Date Posted"].min(), "to", df["Date Posted"].max())
print(f"\nRows with unparseable dates: {df['Date Posted'].isna().sum()}")

#How many reviews per landlord?
print()
print()
print("How many reviews per landlord?")
df["Landlord"].value_counts()

2020-01-07 00:00:00 to 2026-08-18 00:00:00

Rows with unparseable dates: 0


How many reviews per landlord?


C:\Users\afili\AppData\Local\Temp\ipykernel_14524\720901872.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date Posted"] = pd.to_datetime(df["Date Posted"], errors="coerce")


Landlord
Sierra Property Management               235
Breakpointe Coronado                     195
Meridian Group Real Estate Management    132
Koto Group                               124
Wolfe & Associates                       104
Capri IV                                 101
Bartlein & Company                        56
Silverwood Townhomes                      48
State Santa Barbara                        9
Name: count, dtype: int64

**__Let's Do Some Analysis!!__**

In [7]:
# ============================================================
# 1. Review length by sentiment class (overall + per landlord)
# ============================================================

df["word_count"] = df["Review Text"].str.split().str.len()

print("=== Average word count by sentiment (overall) ===")
print(df.groupby("Pos or Neg Review?")["word_count"].agg(["mean", "median", "count"]).round(1))

print("\n=== Average word count by sentiment, per landlord ===")
length_by_landlord_sentiment = df.groupby(["Landlord", "Pos or Neg Review?"])["word_count"].mean().unstack().round(1)
print(length_by_landlord_sentiment)

=== Average word count by sentiment (overall) ===
                     mean  median  count
Pos or Neg Review?                      
Mixed                56.3    53.0     33
Negative            122.4    88.0    429
Positive             37.7    25.0    542

=== Average word count by sentiment, per landlord ===
Pos or Neg Review?                     Mixed  Negative  Positive
Landlord                                                        
Bartlein & Company                      62.0      86.9      22.7
Breakpointe Coronado                    57.8     167.7      37.6
Capri IV                                 NaN     132.8      32.9
Koto Group                              47.0     122.6      40.2
Meridian Group Real Estate Management   67.3      98.2      43.8
Sierra Property Management              68.9     123.5      33.9
Silverwood Townhomes                    23.9     135.5      34.0
State Santa Barbara                      NaN      13.0      28.5
Wolfe & Associates                      

In [8]:
# ============================================================
# 2. Pos/Neg/Mixed proportions (overall + per landlord)
# ============================================================

print("=== Overall Pos/Neg/Mixed counts and percentages ===")
overall_counts = df["Pos or Neg Review?"].value_counts()
overall_pct = df["Pos or Neg Review?"].value_counts(normalize=True).mul(100).round(1)
print(pd.DataFrame({"count": overall_counts, "pct": overall_pct}))

print("\n=== Pos/Neg/Mixed percentages per landlord ===")
landlord_sentiment_pct = df.groupby("Landlord")["Pos or Neg Review?"].value_counts(normalize=True).mul(100).round(1).unstack()
print(landlord_sentiment_pct)

print("\n=== Pos/Neg/Mixed raw counts per landlord ===")
landlord_sentiment_counts = df.groupby("Landlord")["Pos or Neg Review?"].value_counts().unstack(fill_value=0)
print(landlord_sentiment_counts)

=== Overall Pos/Neg/Mixed counts and percentages ===
                    count   pct
Pos or Neg Review?             
Positive              542  54.0
Negative              429  42.7
Mixed                  33   3.3

=== Pos/Neg/Mixed percentages per landlord ===
Pos or Neg Review?                     Mixed  Negative  Positive
Landlord                                                        
Bartlein & Company                       1.8      85.7      12.5
Breakpointe Coronado                     2.1      37.4      60.5
Capri IV                                 NaN      16.8      83.2
Koto Group                               0.8      66.9      32.3
Meridian Group Real Estate Management    2.3      49.2      48.5
Sierra Property Management               3.0      30.2      66.8
Silverwood Townhomes                    16.7      20.8      62.5
State Santa Barbara                      NaN      11.1      88.9
Wolfe & Associates                       8.7      58.7      32.7

=== Pos/Neg/Mixed raw c

In [9]:
# ============================================================
# 3. Cross-tab: Pos/Neg/Mixed label vs. star rating
# ============================================================

print("=== Star rating stats by sentiment label ===")
star_by_sentiment = df.groupby("Pos or Neg Review?")["Stars (if applicable)"].agg(["mean", "median", "min", "max", "count"]).round(2)
print(star_by_sentiment)

print("\n=== Full distribution: sentiment label x star rating ===")
label_star_crosstab = pd.crosstab(df["Pos or Neg Review?"], df["Stars (if applicable)"])
print(label_star_crosstab)

=== Star rating stats by sentiment label ===
                    mean  median  min  max  count
Pos or Neg Review?                               
Mixed               3.27     3.0  2.0  5.0     33
Negative            1.16     1.0  1.0  4.0    417
Positive            4.87     5.0  3.0  5.0    520

=== Full distribution: sentiment label x star rating ===
Stars (if applicable)  1.0  1.2  1.3  2.0  2.5  3.0  3.5  4.0  4.5  4.8  5.0
Pos or Neg Review?                                                          
Mixed                    0    0    0    4    4   11    0   13    0    0    1
Negative               366    1    1   32    0   15    0    2    0    0    0
Positive                 0    0    0    0    0    3    3   51    8    1  454


In [10]:
# ============================================================
# 4. Per-source breakdown: Pos/Neg/Mixed ratios + avg review length
#    (overall and by sentiment, for each source like Google/Yelp/etc.)
# ============================================================

print("=== Pos/Neg/Mixed percentages per source ===")
source_sentiment_pct = df.groupby("Source")["Pos or Neg Review?"].value_counts(normalize=True).mul(100).round(1).unstack()
print(source_sentiment_pct)

print("\n=== Row count per source ===")
print(df["Source"].value_counts())

print("\n=== Average review word count per source (overall) ===")
print(df.groupby("Source")["word_count"].mean().round(1).sort_values(ascending=False))

print("\n=== Average review word count per source, by sentiment ===")
source_length_by_sentiment = df.groupby(["Source", "Pos or Neg Review?"])["word_count"].mean().unstack().round(1)
print(source_length_by_sentiment)

=== Pos/Neg/Mixed percentages per source ===
Pos or Neg Review?    Mixed  Negative  Positive
Source                                         
ApartmentRatings        NaN      40.0      60.0
BBB Complaints          NaN     100.0       NaN
BBB Reviews             NaN     100.0       NaN
Facebook                NaN      15.4      84.6
Google                  2.0      31.6      66.4
Rate My Landlord       28.6      57.1      14.3
RentCafe               21.6       NaN      78.4
Yelp                    1.0      81.2      17.8
Yelp (hidden review)    NaN      71.4      28.6

=== Row count per source ===
Source
Google                  658
Yelp                    197
RentCafe                 37
Rate My Landlord         35
Yelp (hidden review)     28
Facebook                 26
ApartmentRatings         10
BBB Complaints            8
BBB Reviews               5
Name: count, dtype: int64

=== Average review word count per source (overall) ===
Source
BBB Complaints          186.0
Yelp               

In [11]:
# ============================================================
# Most predictive words per class: frequency difference
# (stopwords removed, no lemmatization, min count threshold = 5)
# ============================================================

import re
from collections import Counter
import nltk
from nltk.corpus import stopwords as nltk_stopwords

try:
    nltk_stopwords.words("english")
except LookupError:
    nltk.download("stopwords", quiet=True)

STOPWORDS = set(nltk_stopwords.words("english"))

def tokenize(text):
    words = re.findall(r"[a-z']+", str(text).lower())
    return [w for w in words if w not in STOPWORDS and len(w) > 2]

# Split reviews by class (excluding Mixed for this comparison — pure Pos vs Neg contrast)
pos_text = df[df["Pos or Neg Review?"] == "Positive"]["Review Text"]
neg_text = df[df["Pos or Neg Review?"] == "Negative"]["Review Text"]

pos_words = [w for text in pos_text for w in tokenize(text)]
neg_words = [w for text in neg_text for w in tokenize(text)]

pos_counts = Counter(pos_words)
neg_counts = Counter(neg_words)

pos_total = sum(pos_counts.values())
neg_total = sum(neg_counts.values())

# Build frequency-difference table for words meeting the min count threshold
MIN_COUNT = 5
all_words = set(pos_counts) | set(neg_counts)

rows = []
for word in all_words:
    total_count = pos_counts[word] + neg_counts[word]
    if total_count < MIN_COUNT:
        continue
    pos_freq = pos_counts[word] / pos_total  # normalized rate within Positive
    neg_freq = neg_counts[word] / neg_total  # normalized rate within Negative
    diff = pos_freq - neg_freq
    rows.append({
        "word": word,
        "pos_count": pos_counts[word],
        "neg_count": neg_counts[word],
        "pos_freq_pct": round(pos_freq * 100, 4),
        "neg_freq_pct": round(neg_freq * 100, 4),
        "diff": diff
    })

word_diff_df = pd.DataFrame(rows)

print(f"Words meeting min count threshold ({MIN_COUNT}+): {len(word_diff_df)}")

print("\n=== Top 25 words most associated with POSITIVE reviews ===")
print(word_diff_df.sort_values("diff", ascending=False).head(25)[["word", "pos_count", "neg_count", "diff"]].to_string(index=False))

print("\n=== Top 25 words most associated with NEGATIVE reviews ===")
print(word_diff_df.sort_values("diff", ascending=True).head(25)[["word", "pos_count", "neg_count", "diff"]].to_string(index=False))

Words meeting min count threshold (5+): 1321

=== Top 25 words most associated with POSITIVE reviews ===
           word  pos_count  neg_count     diff
          great        174         14 0.015880
        helpful        128         10 0.011693
       positive         93          5 0.008585
          staff         90         33 0.007203
professionalism         84         28 0.006833
 responsiveness         75         16 0.006454
       property        162        228 0.006350
         always         72         21 0.005974
           nice         70         20 0.005825
      recommend         73         34 0.005559
       friendly         59          5 0.005375
         sierra         78         52 0.005324
          thank         57          5 0.005186
     experience         83         76 0.004855
           easy         53          5 0.004808
          super         54          9 0.004746
           best         55         12 0.004722
   professional         55         16 0.004565
  

In [12]:
# ============================================================
# Naive Bayes classifier (adapted from LING 110 Assignment 4, Task 2)
# Binary: Positive vs Negative (Mixed excluded), 80/20 train/test split
# ============================================================

import re
import math
from collections import Counter
import random

# --- Prep: lowercase + strip punctuation, keep Pos/Neg only ---
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^\w\s]", " ", text)  # strip punctuation
    return text.split()

nb_df = df[df["Pos or Neg Review?"].isin(["Positive", "Negative"])].copy()
nb_df["tokens"] = nb_df["Review Text"].apply(clean_text)

# --- 80/20 train/test split (shuffled, reproducible) ---
random.seed(0)
indices = list(nb_df.index)
random.shuffle(indices)
split_point = int(len(indices) * 0.8)
train_idx = indices[:split_point]
test_idx = indices[split_point:]

train_df = nb_df.loc[train_idx]
test_df = nb_df.loc[test_idx]

print(f"Train size: {len(train_df)}  |  Test size: {len(test_df)}")
print(train_df["Pos or Neg Review?"].value_counts())

# --- define_vocab(): unique words across all TRAINING reviews ---
def define_vocab(token_lists):
    vocab = set()
    for tokens in token_lists:
        vocab.update(tokens)
    return vocab

vocab = define_vocab(train_df["tokens"])
print(f"\nVocab size: {len(vocab)}")

# --- bag_of_words(): word counts for a single document, restricted to vocab ---
def bag_of_words(tokens, vocab=None):
    if vocab is not None:
        tokens = [t for t in tokens if t in vocab]
    return Counter(tokens)

# --- get_priors(): log P(c) for each class ---
def get_priors(labels):
    counts = Counter(labels)
    total = len(labels)
    return {c: math.log(count / total) for c, count in counts.items()}

priors = get_priors(train_df["Pos or Neg Review?"])
print(f"\nPriors: {priors}")

# --- get_likelihoods(): add-1 smoothed log P(word|c) for every word in vocab, per class ---
def get_likelihoods(train_df, vocab):
    likelihoods = {}
    for class_label, group in train_df.groupby("Pos or Neg Review?"):
        class_counts = Counter()
        for tokens in group["tokens"]:
            class_counts.update([t for t in tokens if t in vocab])
        class_counts.update(vocab)  # add-1 smoothing: every vocab word starts at count 1
        total_words_in_class = sum(class_counts.values())
        likelihoods[class_label] = {
            word: math.log(count / total_words_in_class)
            for word, count in class_counts.items()
        }
    return likelihoods

likelihoods = get_likelihoods(train_df, vocab)

# --- get_class_score(): prior + sum(count * log-likelihood) ---
def get_class_score(features, class_label, priors, likelihoods):
    score = priors[class_label]
    for word, count in features.items():
        if word in likelihoods[class_label]:
            score += count * likelihoods[class_label][word]
    return score

# --- classify(): pick highest-scoring class ---
def classify(tokens, vocab, priors, likelihoods):
    features = bag_of_words(tokens, vocab)
    scores = {c: get_class_score(features, c, priors, likelihoods) for c in priors}
    return max(scores, key=scores.get)

# --- Run on test set, build confusion matrix, compute precision/recall/F1 ---
test_df = test_df.copy()
test_df["predicted"] = test_df["tokens"].apply(lambda t: classify(t, vocab, priors, likelihoods))

confusion = pd.crosstab(test_df["Pos or Neg Review?"], test_df["predicted"], rownames=["Actual"], colnames=["Predicted"])
print("\n=== Confusion Matrix ===")
print(confusion)

tp = confusion.loc["Positive", "Positive"] if "Positive" in confusion.index and "Positive" in confusion.columns else 0
fp = confusion.loc["Negative", "Positive"] if "Negative" in confusion.index and "Positive" in confusion.columns else 0
fn = confusion.loc["Positive", "Negative"] if "Positive" in confusion.index and "Negative" in confusion.columns else 0

precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

accuracy = (test_df["Pos or Neg Review?"] == test_df["predicted"]).mean()

print(f"\nAccuracy:  {accuracy:.4f}")
print(f"Precision (Positive class): {precision:.4f}")
print(f"Recall (Positive class):    {recall:.4f}")
print(f"F1 (Positive class):        {f1:.4f}")

# --- Most predictive words per class: log P(word|Positive) - log P(word|Negative) ---
word_diffs = []
for word in vocab:
    pos_ll = likelihoods["Positive"].get(word)
    neg_ll = likelihoods["Negative"].get(word)
    if pos_ll is not None and neg_ll is not None:
        word_diffs.append((word, pos_ll - neg_ll))

word_diffs_df = pd.DataFrame(word_diffs, columns=["word", "diff"]).sort_values("diff", ascending=False)

print("\n=== Top 20 words most predictive of POSITIVE (Naive Bayes) ===")
print(word_diffs_df.head(20).to_string(index=False))

print("\n=== Top 20 words most predictive of NEGATIVE (Naive Bayes) ===")
print(word_diffs_df.tail(20).sort_values("diff").to_string(index=False))



Train size: 776  |  Test size: 195
Pos or Neg Review?
Positive    426
Negative    350
Name: count, dtype: int64

Vocab size: 4798

Priors: {'Positive': -0.5997131739138483, 'Negative': -0.7962193656997594}

=== Confusion Matrix ===
Predicted  Negative  Positive
Actual                       
Negative         76         3
Positive          3       113

Accuracy:  0.9692
Precision (Positive class): 0.9741
Recall (Positive class):    0.9741
F1 (Positive class):        0.9741

=== Top 20 words most predictive of POSITIVE (Naive Bayes) ===
      word     diff
 excellent 3.695215
   helpful 3.667044
   amazing 3.638057
   awesome 3.577432
     brett 3.545683
    prompt 3.369793
   quickly 3.330572
    cayden 3.289750
  positive 3.275764
     thank 3.247190
     great 3.209707
  friendly 3.165697
  brooklyn 3.107428
     deedy 3.107428
 wonderful 3.107428
      easy 3.107428
      love 3.002068
   excited 3.002068
      dave 2.884285
incredible 2.884285

=== Top 20 words most predictive of NEG

In [13]:
# ============================================================
# Error analysis: pull up the 6 misclassified test reviews
# ============================================================

misclassified = test_df[test_df["Pos or Neg Review?"] != test_df["predicted"]].copy()

for idx, row in misclassified.iterrows():
    features = bag_of_words(row["tokens"], vocab)
    pos_score = get_class_score(features, "Positive", priors, likelihoods)
    neg_score = get_class_score(features, "Negative", priors, likelihoods)

    print(f"Landlord: {row['Landlord']}")
    print(f"Actual: {row['Pos or Neg Review?']}  |  Predicted: {row['predicted']}")
    print(f"Positive score: {pos_score:.3f}  |  Negative score: {neg_score:.3f}")
    print(f"Review: {row['Review Text']}")
    print("-" * 80)

Landlord: Wolfe & Associates
Actual: Negative  |  Predicted: Positive
Positive score: -34.737  |  Negative score: -38.918
Review: Negative
Responsiveness, Quality, Professionalism, Value
--------------------------------------------------------------------------------
Landlord: Wolfe & Associates
Actual: Negative  |  Predicted: Positive
Positive score: -57.288  |  Negative score: -60.720
Review: lame
#Fair Pricing
#Student-Friendly
#Hidden Fees
#Overpriced
--------------------------------------------------------------------------------
Landlord: Wolfe & Associates
Actual: Positive  |  Predicted: Negative
Positive score: -449.461  |  Negative score: -441.665
Review: Renting with Wolfe and Associates was great. They were very accommodating and understanding in the move in and move out process. They communicated clearly and were easy to get ahold of. If you let them know of anything ahead of time Scott will take care of it. They returned most of our security deposit with no hassle and clea

In [14]:
# ============================================================
# Per-landlord Naive Bayes: WOLFE & ASSOCIATES
# ============================================================

landlord_name = "Wolfe & Associates"
landlord_df = nb_df[nb_df["Landlord"] == landlord_name].copy()

print(f"Total {landlord_name} reviews (Pos/Neg only): {len(landlord_df)}")
print(landlord_df["Pos or Neg Review?"].value_counts())

# --- 80/20 split ---
random.seed(0)
indices = list(landlord_df.index)
random.shuffle(indices)
split_point = int(len(indices) * 0.8)
train_idx = indices[:split_point]
test_idx = indices[split_point:]

train_l = landlord_df.loc[train_idx]
test_l = landlord_df.loc[test_idx]

print(f"\nTrain size: {len(train_l)}  |  Test size: {len(test_l)}")

# --- Vocab, priors, likelihoods (reusing the same functions from before) ---
vocab_l = define_vocab(train_l["tokens"])
priors_l = get_priors(train_l["Pos or Neg Review?"])
likelihoods_l = get_likelihoods(train_l, vocab_l)

print(f"Vocab size: {len(vocab_l)}")

# --- Classify test set, confusion matrix, metrics ---
test_l = test_l.copy()
test_l["predicted"] = test_l["tokens"].apply(lambda t: classify(t, vocab_l, priors_l, likelihoods_l))

confusion_l = pd.crosstab(test_l["Pos or Neg Review?"], test_l["predicted"], rownames=["Actual"], colnames=["Predicted"])
print("\n=== Confusion Matrix ===")
print(confusion_l)

accuracy_l = (test_l["Pos or Neg Review?"] == test_l["predicted"]).mean()
print(f"\nAccuracy: {accuracy_l:.4f}")

# --- Top 10 predictive words per class ---
word_diffs_l = []
for word in vocab_l:
    pos_ll = likelihoods_l.get("Positive", {}).get(word)
    neg_ll = likelihoods_l.get("Negative", {}).get(word)
    if pos_ll is not None and neg_ll is not None:
        word_diffs_l.append((word, pos_ll - neg_ll))

word_diffs_l_df = pd.DataFrame(word_diffs_l, columns=["word", "diff"]).sort_values("diff", ascending=False)

print(f"\n=== Top 10 words most predictive of POSITIVE for {landlord_name} ===")
print(word_diffs_l_df.head(10).to_string(index=False))

print(f"\n=== Top 10 words most predictive of NEGATIVE for {landlord_name} ===")
print(word_diffs_l_df.tail(10).sort_values("diff").to_string(index=False))

Total Wolfe & Associates reviews (Pos/Neg only): 95
Pos or Neg Review?
Negative    61
Positive    34
Name: count, dtype: int64

Train size: 76  |  Test size: 19
Vocab size: 1526

=== Confusion Matrix ===
Predicted  Negative  Positive
Actual                       
Negative         15         0
Positive          2         2

Accuracy: 0.8947

=== Top 10 words most predictive of POSITIVE for Wolfe & Associates ===
     word     diff
     easy 2.555479
  amazing 2.373158
   fridge 2.373158
following 2.150014
  pleased 2.150014
 positive 2.150014
impressed 2.150014
     fast 2.150014
  plumber 2.150014
   expert 2.150014

=== Top 10 words most predictive of NEGATIVE for Wolfe & Associates ===
    word      diff
   lease -2.371774
    only -2.069494
     did -2.008869
 charges -1.875337
     not -1.801230
  charge -1.721187
    cost -1.721187
  people -1.721187
    also -1.721187
cleaning -1.634175


In [15]:
# ============================================================
# Per-landlord Naive Bayes: SIERRA PROPERTY MANAGEMENT
# ============================================================

landlord_name = "Sierra Property Management"
landlord_df = nb_df[nb_df["Landlord"] == landlord_name].copy()

print(f"Total {landlord_name} reviews (Pos/Neg only): {len(landlord_df)}")
print(landlord_df["Pos or Neg Review?"].value_counts())

# --- 80/20 split ---
random.seed(0)
indices = list(landlord_df.index)
random.shuffle(indices)
split_point = int(len(indices) * 0.8)
train_idx = indices[:split_point]
test_idx = indices[split_point:]

train_l = landlord_df.loc[train_idx]
test_l = landlord_df.loc[test_idx]

print(f"\nTrain size: {len(train_l)}  |  Test size: {len(test_l)}")

# --- Vocab, priors, likelihoods ---
vocab_l = define_vocab(train_l["tokens"])
priors_l = get_priors(train_l["Pos or Neg Review?"])
likelihoods_l = get_likelihoods(train_l, vocab_l)

print(f"Vocab size: {len(vocab_l)}")

# --- Classify test set, confusion matrix, metrics ---
test_l = test_l.copy()
test_l["predicted"] = test_l["tokens"].apply(lambda t: classify(t, vocab_l, priors_l, likelihoods_l))

confusion_l = pd.crosstab(test_l["Pos or Neg Review?"], test_l["predicted"], rownames=["Actual"], colnames=["Predicted"])
print("\n=== Confusion Matrix ===")
print(confusion_l)

accuracy_l = (test_l["Pos or Neg Review?"] == test_l["predicted"]).mean()
print(f"\nAccuracy: {accuracy_l:.4f}")

# --- Top 10 predictive words per class ---
word_diffs_l = []
for word in vocab_l:
    pos_ll = likelihoods_l.get("Positive", {}).get(word)
    neg_ll = likelihoods_l.get("Negative", {}).get(word)
    if pos_ll is not None and neg_ll is not None:
        word_diffs_l.append((word, pos_ll - neg_ll))

word_diffs_l_df = pd.DataFrame(word_diffs_l, columns=["word", "diff"]).sort_values("diff", ascending=False)

print(f"\n=== Top 10 words most predictive of POSITIVE for {landlord_name} ===")
print(word_diffs_l_df.head(10).to_string(index=False))

print(f"\n=== Top 10 words most predictive of NEGATIVE for {landlord_name} ===")
print(word_diffs_l_df.tail(10).sort_values("diff").to_string(index=False))

Total Sierra Property Management reviews (Pos/Neg only): 228
Pos or Neg Review?
Positive    157
Negative     71
Name: count, dtype: int64

Train size: 182  |  Test size: 46
Vocab size: 2076

=== Confusion Matrix ===
Predicted  Negative  Positive
Actual                       
Negative          8         2
Positive          2        34

Accuracy: 0.9130

=== Top 10 words most predictive of POSITIVE for Sierra Property Management ===
          word     diff
      positive 4.203328
        robert 3.356030
         quick 3.238247
       quickly 3.030608
        jovien 2.950565
        timely 2.863554
responsiveness 2.690282
        prompt 2.662883
  professional 2.662883
        always 2.575872

=== Top 10 words most predictive of NEGATIVE for Sierra Property Management ===
          word      diff
         which -2.367555
        months -2.306930
      horrible -2.173399
          rude -2.019248
        broken -2.019248
         these -2.019248
          told -2.019248
       deposit -1.97

In [16]:
# ============================================================
# Per-landlord Naive Bayes: MERIDIAN GROUP REAL ESTATE MANAGEMENT
# ============================================================

landlord_name = "Meridian Group Real Estate Management"
landlord_df = nb_df[nb_df["Landlord"] == landlord_name].copy()

print(f"Total {landlord_name} reviews (Pos/Neg only): {len(landlord_df)}")
print(landlord_df["Pos or Neg Review?"].value_counts())

# --- 80/20 split ---
random.seed(0)
indices = list(landlord_df.index)
random.shuffle(indices)
split_point = int(len(indices) * 0.8)
train_idx = indices[:split_point]
test_idx = indices[split_point:]

train_l = landlord_df.loc[train_idx]
test_l = landlord_df.loc[test_idx]

print(f"\nTrain size: {len(train_l)}  |  Test size: {len(test_l)}")

# --- Vocab, priors, likelihoods ---
vocab_l = define_vocab(train_l["tokens"])
priors_l = get_priors(train_l["Pos or Neg Review?"])
likelihoods_l = get_likelihoods(train_l, vocab_l)

print(f"Vocab size: {len(vocab_l)}")

# --- Classify test set, confusion matrix, metrics ---
test_l = test_l.copy()
test_l["predicted"] = test_l["tokens"].apply(lambda t: classify(t, vocab_l, priors_l, likelihoods_l))

confusion_l = pd.crosstab(test_l["Pos or Neg Review?"], test_l["predicted"], rownames=["Actual"], colnames=["Predicted"])
print("\n=== Confusion Matrix ===")
print(confusion_l)

accuracy_l = (test_l["Pos or Neg Review?"] == test_l["predicted"]).mean()
print(f"\nAccuracy: {accuracy_l:.4f}")

# --- Top 10 predictive words per class ---
word_diffs_l = []
for word in vocab_l:
    pos_ll = likelihoods_l.get("Positive", {}).get(word)
    neg_ll = likelihoods_l.get("Negative", {}).get(word)
    if pos_ll is not None and neg_ll is not None:
        word_diffs_l.append((word, pos_ll - neg_ll))

word_diffs_l_df = pd.DataFrame(word_diffs_l, columns=["word", "diff"]).sort_values("diff", ascending=False)

print(f"\n=== Top 10 words most predictive of POSITIVE for {landlord_name} ===")
print(word_diffs_l_df.head(10).to_string(index=False))

print(f"\n=== Top 10 words most predictive of NEGATIVE for {landlord_name} ===")
print(word_diffs_l_df.tail(10).sort_values("diff").to_string(index=False))

Total Meridian Group Real Estate Management reviews (Pos/Neg only): 129
Pos or Neg Review?
Negative    65
Positive    64
Name: count, dtype: int64

Train size: 103  |  Test size: 26
Vocab size: 1630

=== Confusion Matrix ===
Predicted  Negative  Positive
Actual                       
Negative         11         0
Positive          1        14

Accuracy: 0.9615

=== Top 10 words most predictive of POSITIVE for Meridian Group Real Estate Management ===
        word     diff
  responsive 2.925267
professional 2.925267
       deedy 2.724596
       staff 2.606813
        best 2.606813
        kind 2.473282
       super 2.473282
  appreciate 2.473282
    positive 2.319131
    friendly 2.319131

=== Top 10 words most predictive of NEGATIVE for Meridian Group Real Estate Management ===
 word      diff
 your -2.804833
 mold -2.180679
  don -2.180679
 take -2.180679
 fees -2.180679
   us -2.180679
money -2.111686
water -2.111686
  pay -1.957535
 said -1.957535


In [17]:
# ============================================================
# Per-landlord Naive Bayes: KOTO GROUP
# ============================================================

landlord_name = "Koto Group"
landlord_df = nb_df[nb_df["Landlord"] == landlord_name].copy()

print(f"Total {landlord_name} reviews (Pos/Neg only): {len(landlord_df)}")
print(landlord_df["Pos or Neg Review?"].value_counts())

# --- 80/20 split ---
random.seed(0)
indices = list(landlord_df.index)
random.shuffle(indices)
split_point = int(len(indices) * 0.8)
train_idx = indices[:split_point]
test_idx = indices[split_point:]

train_l = landlord_df.loc[train_idx]
test_l = landlord_df.loc[test_idx]

print(f"\nTrain size: {len(train_l)}  |  Test size: {len(test_l)}")

# --- Vocab, priors, likelihoods ---
vocab_l = define_vocab(train_l["tokens"])
priors_l = get_priors(train_l["Pos or Neg Review?"])
likelihoods_l = get_likelihoods(train_l, vocab_l)

print(f"Vocab size: {len(vocab_l)}")

# --- Classify test set, confusion matrix, metrics ---
test_l = test_l.copy()
test_l["predicted"] = test_l["tokens"].apply(lambda t: classify(t, vocab_l, priors_l, likelihoods_l))

confusion_l = pd.crosstab(test_l["Pos or Neg Review?"], test_l["predicted"], rownames=["Actual"], colnames=["Predicted"])
print("\n=== Confusion Matrix ===")
print(confusion_l)

accuracy_l = (test_l["Pos or Neg Review?"] == test_l["predicted"]).mean()
print(f"\nAccuracy: {accuracy_l:.4f}")

# --- Top 10 predictive words per class ---
word_diffs_l = []
for word in vocab_l:
    pos_ll = likelihoods_l.get("Positive", {}).get(word)
    neg_ll = likelihoods_l.get("Negative", {}).get(word)
    if pos_ll is not None and neg_ll is not None:
        word_diffs_l.append((word, pos_ll - neg_ll))

word_diffs_l_df = pd.DataFrame(word_diffs_l, columns=["word", "diff"]).sort_values("diff", ascending=False)

print(f"\n=== Top 10 words most predictive of POSITIVE for {landlord_name} ===")
print(word_diffs_l_df.head(10).to_string(index=False))

print(f"\n=== Top 10 words most predictive of NEGATIVE for {landlord_name} ===")
print(word_diffs_l_df.tail(10).sort_values("diff").to_string(index=False))

Total Koto Group reviews (Pos/Neg only): 123
Pos or Neg Review?
Negative    83
Positive    40
Name: count, dtype: int64

Train size: 98  |  Test size: 25
Vocab size: 1755

=== Confusion Matrix ===
Predicted  Negative  Positive
Actual                       
Negative         16         0
Positive          2         7

Accuracy: 0.9200

=== Top 10 words most predictive of POSITIVE for Koto Group ===
          word     diff
      positive 3.462828
      friendly 3.157446
          fair 3.003295
         quick 2.820974
    responsive 2.715613
responsiveness 2.677873
      security 2.597830
        during 2.597830
        prompt 2.597830
           pet 2.597830

=== Top 10 words most predictive of NEGATIVE for Koto Group ===
 word      diff
 this -2.269704
    t -2.071879
  don -2.007340
email -2.007340
there -1.945465
  who -1.879507
other -1.732903
phone -1.732903
asked -1.732903
never -1.732903


In [18]:
# ============================================================
# Logistic regression (SGDClassifier, same setup as LING 110 Assignment 4, Task 1)
# Full dataset: Positive vs Negative (Mixed excluded), 80/20 split
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import precision_recall_fscore_support
import numpy as np

np.random.seed(0)

# --- Reuse the same train/test split as the full-dataset Naive Bayes ---
lr_df = df[df["Pos or Neg Review?"].isin(["Positive", "Negative"])].copy()

random.seed(0)
indices = list(lr_df.index)
random.shuffle(indices)
split_point = int(len(indices) * 0.8)
train_idx = indices[:split_point]
test_idx = indices[split_point:]

train_lr = lr_df.loc[train_idx]
test_lr = lr_df.loc[test_idx]

print(f"Train size: {len(train_lr)}  |  Test size: {len(test_lr)}")

# --- TF-IDF vectorizer, same as your assignment (just using raw text lists instead of filenames) ---
vectorizer = TfidfVectorizer()

train_features = vectorizer.fit_transform(train_lr["Review Text"])
test_features = vectorizer.transform(test_lr["Review Text"])

# --- Numerical class labels: Positive = 1, Negative = 0 ---
train_classes = train_lr["Pos or Neg Review?"].map({"Negative": 0, "Positive": 1}).values
test_classes = test_lr["Pos or Neg Review?"].map({"Negative": 0, "Positive": 1}).values

vocab = vectorizer.get_feature_names_out()
print(f"Vocab size: {len(vocab)}")

# --- Train the classifier, same core arguments as your assignment ---
simple_classifier = SGDClassifier(loss="log_loss", random_state=0, learning_rate="constant", eta0=1)
simple_classifier.fit(train_features, train_classes)

# --- Accuracy ---
accuracy = simple_classifier.score(test_features, test_classes)
print(f"\nAccuracy: {accuracy:.4f}")

# --- Precision, recall, F1 ---
test_predictions = simple_classifier.predict(test_features)
precision, recall, f1, _ = precision_recall_fscore_support(test_classes, test_predictions, average="binary")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1:        {f1:.4f}")

# --- Top 10 words most associated with each class (same method as Task 1.1) ---
coefs = simple_classifier.coef_[0]
labeled_coefs = pd.Series(coefs, index=vocab)
labeled_coefs.sort_values(ascending=False, inplace=True)

print("\n=== Top 10 words most associated with POSITIVE ===")
print(labeled_coefs.head(10))

print("\n=== Top 10 words most associated with NEGATIVE ===")
print(labeled_coefs.tail(10))

Train size: 776  |  Test size: 195
Vocab size: 4770

Accuracy: 0.9692
Precision: 0.9741
Recall:    0.9741
F1:        0.9741

=== Top 10 words most associated with POSITIVE ===
great       5.851463
positive    5.320816
helpful     4.771760
best        4.201654
easy        3.958943
nice        3.858165
thank       3.609409
sierra      3.451672
staff       3.291312
so          3.280823
dtype: float64

=== Top 10 words most associated with NEGATIVE ===
never      -3.155361
terrible   -3.456976
poor       -3.610798
no         -3.787447
this       -4.087153
rude       -4.265785
they       -4.395814
worst      -4.495083
negative   -4.917827
not        -5.803611
dtype: float64
